# 04 — Generate orders

**Business objective:** this is the core fact table, and it's where we
deliberately engineer the Sydney revenue drop the whole MVP is built to
investigate. Everything downstream (order_items, the eventual ML trend model,
the Streamlit explanation) traces back to the order generation logic here.

**What we're generating:** ~50,000 orders over an 18-month window, linked to
customers and stores, with a **deterministic, seeded reduction in Sydney order
volume during the final 2 months**.

In [1]:
import sys
sys.path.insert(0, '..')
from notebooks import nb_config as cfg

import pandas as pd
import numpy as np

rng = np.random.default_rng(cfg.SEED + 3)

customers = pd.read_csv(cfg.DATA_DIR / "customers.csv")
stores = pd.read_csv(cfg.DATA_DIR / "stores.csv")
print(f"Loaded {len(customers):,} customers and {len(stores):,} stores")

Loaded 5,000 customers and 8 stores


## Generation logic — the engineered anomaly

We simulate order volume day-by-day per store. Each store has a baseline daily
order rate. For **every store except the anomaly window**, that rate is stable
with normal noise. For **Sydney stores in the final `ANOMALY_MONTHS`**, the
rate is multiplied by `(1 - ANOMALY_ORDER_REDUCTION)`.

This is a deliberate, reproducible rule — not injected noise — so that later,
when the AI assistant "investigates" the drop, there is a real, traceable
cause in the data (fewer orders generated, not random variance).

In [2]:
end_date = pd.Timestamp("2026-07-01")
start_date = end_date - pd.DateOffset(months=cfg.ORDER_HISTORY_MONTHS)
anomaly_start = end_date - pd.DateOffset(months=cfg.ANOMALY_MONTHS)

print(f"Order history: {start_date.date()} to {end_date.date()}")
print(f"Anomaly window: {anomaly_start.date()} to {end_date.date()} (Sydney only)")

dates = pd.date_range(start_date, end_date, freq="D")

# --- Second design fix, not just the signup-eligibility bug ---
# A first pass kept a top-down "store capacity" model (Poisson orders per store
# per day) and only reweighted WHICH customer got attributed to an order during
# the Sydney anomaly. That reshuffles the mix but never reduces total volume --
# retention decline needs to reduce how OFTEN existing customers buy, which only
# shows up if order volume is built bottom-up from individual customers.
#
# New model: each eligible customer has a small daily probability of placing an
# order. Sydney customers in the anomaly window get that probability reduced --
# more so if they're an "existing" customer (retention decline) than a "new"
# one, plus a small blanket softness applied to everyone in Sydney (general
# demand). Order volume is now the direct sum of individual customer behaviour,
# so a retention penalty genuinely lowers total Sydney revenue.
BASE_DAILY_ORDER_PROB = 0.018  # ~1.8%/day per eligible customer, ~10 orders/customer over 18 months

customer_ids_arr = customers["customer_id"].to_numpy()
customer_city_arr = customers["city"].to_numpy()
signup_dates_arr = pd.to_datetime(customers["signup_date"]).to_numpy()

city_to_stores = stores.groupby("city")["store_id"].apply(list).to_dict()

order_rows = []
order_id = 1
for d in dates:
    d64 = np.datetime64(d)
    eligible_mask = signup_dates_arr <= d64
    if not eligible_mask.any():
        continue

    ids = customer_ids_arr[eligible_mask]
    cities_today = customer_city_arr[eligible_mask]
    signups_today = signup_dates_arr[eligible_mask]

    probs = np.full(len(ids), BASE_DAILY_ORDER_PROB)

    is_sydney = cities_today == cfg.ANOMALY_CITY
    if d >= anomaly_start and is_sydney.any():
        probs[is_sydney] *= (1 - cfg.ANOMALY_BASE_SOFTNESS)          # general softness, everyone
        existing_cutoff = np.datetime64(d - pd.Timedelta(days=cfg.ANOMALY_EXISTING_CUTOFF_DAYS))
        is_existing = signups_today <= existing_cutoff
        probs[is_sydney & is_existing] *= (1 - cfg.ANOMALY_RETENTION_PENALTY)  # extra hit, existing custs only

    ordered_today = rng.random(len(ids)) < probs
    if not ordered_today.any():
        continue

    for cid, city in zip(ids[ordered_today], cities_today[ordered_today]):
        store_id = rng.choice(city_to_stores[city])
        order_rows.append({
            "order_id": order_id,
            "customer_id": int(cid),
            "store_id": int(store_id),
            "order_date": d,
            "channel": rng.choice(["online", "in_store"], p=[0.55, 0.45]),
            "order_status": rng.choice(["completed", "cancelled"], p=[0.95, 0.05]),
        })
        order_id += 1

orders = pd.DataFrame(order_rows)
print(f"Generated {len(orders):,} orders")
orders.head()


Order history: 2025-01-01 to 2026-07-01
Anomaly window: 2026-05-01 to 2026-07-01 (Sydney only)


Generated 35,591 orders


,order_id,customer_id,store_id,order_date,channel,order_status
0,1,108,3,2025-01-01,online,completed
1,2,214,8,2025-01-01,online,completed
2,3,398,5,2025-01-01,online,completed
3,4,481,5,2025-01-01,online,completed
4,5,575,7,2025-01-01,in_store,completed


## Sanity check: does the anomaly actually show up?

In [3]:
orders["order_date"] = pd.to_datetime(orders["order_date"])
monthly = (
    orders.merge(stores[["store_id", "city"]], on="store_id")
    .assign(month=lambda d: d["order_date"].dt.to_period("M"))
    .groupby(["month", "city"])
    .size()
    .unstack(fill_value=0)
)
monthly[["Sydney", "Melbourne"]].tail(6)

city,Sydney,Melbourne
month,,
2026-02,378,341
2026-03,429,423
2026-04,410,427
2026-05,198,449
2026-06,247,419
2026-07,6,18


Sydney's order count should visibly drop in the last two months relative to Melbourne, which has no engineered anomaly.

## Validation

In [4]:
assert orders["order_id"].is_unique
assert orders["customer_id"].between(1, len(customers)).all()
assert orders["store_id"].isin(stores["store_id"]).all()
assert orders.isnull().sum().sum() == 0
assert len(orders) > 30_000, "order volume looks too low for an 18-month window"
# The check that actually proves the bug is fixed:
_check = orders.merge(customers[["customer_id", "signup_date"]], on="customer_id")
_violations = (pd.to_datetime(_check["order_date"]) < pd.to_datetime(_check["signup_date"])).sum()
assert _violations == 0, f"{_violations} orders occur before the customer's signup date!"
print("Confirmed: zero orders precede their customer's signup date.")

print("All validation checks passed.")

Confirmed: zero orders precede their customer's signup date.
All validation checks passed.


In [5]:
out_path = cfg.DATA_DIR / "orders.csv"
orders.to_csv(out_path, index=False)
print(f"Wrote {len(orders):,} rows to {out_path}")

Wrote 35,591 rows to /home/claude/project/eaida/data/raw/orders.csv


## Summary

Generated an 18-month order history with a deliberate, seeded ~38% reduction
in Sydney order volume over the final 2 months, confirmed visually against
Melbourne's stable trend. This table is the anchor for the MVP's target
question. Saved to `data/raw/orders.csv`.